In [ ]:
import time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split, Subset, ConcatDataset
import matplotlib.pyplot as plt
from skimage.metrics import structural_similarity as ssim
import random
import gc

import os, json, csv
from itertools import combinations
from collections import defaultdict

import piq

SEED=0
torch.manual_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)

In [ ]:
class NumpyDataset(Dataset):
    def __init__(self, x_path, y_path):
        self.x = np.load(x_path)
        self.y = np.load(y_path)
    
    def __len__(self):
        return self.x.shape[0]
    
    def __getitem__(self, idx):
        x_full = self.x[idx]
        main_x = torch.tensor(x_full[:11, :, :], dtype=torch.float32)
        time_enc = torch.tensor(x_full[11:12, :, :], dtype=torch.float32)
        y = torch.tensor(self.y[idx], dtype=torch.float32)
        return main_x, time_enc, y

class ConvBlock(nn.Module):
    def __init__(self, in_channels, time_channels, out_channels, kernel_size=3, activation=nn.ReLU()):
        super(ConvBlock, self).__init__()
        padding = kernel_size // 2  # same padding for odd kernel sizes
        self.activation = activation
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size, padding=padding)
        self.time_conv = nn.Conv2d(time_channels, out_channels, kernel_size, padding=padding)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size, padding=padding)
        
    def forward(self, x, time_emb):
        # Resize time embedding if its spatial dimensions differ from x
        if time_emb.shape[-2:] != x.shape[-2:]:
            time_emb = F.interpolate(time_emb, size=x.shape[-2:], mode='bilinear', align_corners=False)
        out_x = self.activation(self.conv1(x))
        out_time = self.activation(self.time_conv(time_emb))
        out = out_x + out_time
        out = self.activation(self.conv2(out))
        return out

class EncoderBlock(nn.Module):
    def __init__(self, in_channels, time_channels, out_channels):
        super(EncoderBlock, self).__init__()
        self.conv_block = ConvBlock(in_channels, time_channels, out_channels)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        
    def forward(self, x, time_emb):
        x_conv = self.conv_block(x, time_emb)
        x_pooled = self.pool(x_conv)
        return x_conv, x_pooled

def crop_tensor(skip, target):
    diffY = skip.size(2) - target.size(2)
    diffX = skip.size(3) - target.size(3)
    return skip[:, :, diffY//2 : skip.size(2) - (diffY - diffY//2), 
                   diffX//2 : skip.size(3) - (diffX - diffX//2)]

class DecoderBlock(nn.Module):
    def __init__(self, in_channels, skip_channels, time_channels, out_channels):
        super(DecoderBlock, self).__init__()
        self.upsample = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False)
        self.conv_block = ConvBlock(in_channels + skip_channels, time_channels, out_channels)
        
    def forward(self, x, skip, time_emb):
        x = self.upsample(x)
        if x.size(2) != skip.size(2) or x.size(3) != skip.size(3):
            skip = crop_tensor(skip, x)
        x = torch.cat([x, skip], dim=1)
        x = self.conv_block(x, time_emb)
        return x

class ConditionalUNet(nn.Module):
    def __init__(self, input_channels=11, time_channels=1, base_filters=64):
        super(ConditionalUNet, self).__init__()
        # Encoder path
        self.enc1 = EncoderBlock(input_channels, time_channels, base_filters)          # 8 -> 64
        self.enc2 = EncoderBlock(base_filters, time_channels, base_filters * 2)          # 64 -> 128
        self.enc3 = EncoderBlock(base_filters * 2, time_channels, base_filters * 4)      # 128 -> 256
        self.enc4 = EncoderBlock(base_filters * 4, time_channels, base_filters * 8)      # 256 -> 512
        
        self.bottleneck = ConvBlock(base_filters * 8, time_channels, base_filters * 16)  # 512 -> 1024
        
        self.dec1 = DecoderBlock(base_filters * 16, base_filters * 8, time_channels, base_filters * 8)  # (1024+512) -> 512
        self.dec2 = DecoderBlock(base_filters * 8, base_filters * 4, time_channels, base_filters * 4)   # (512+256) -> 256
        self.dec3 = DecoderBlock(base_filters * 4, base_filters * 2, time_channels, base_filters * 2)   # (256+128) -> 128
        self.dec4 = DecoderBlock(base_filters * 2, base_filters, time_channels, base_filters)           # (128+64) -> 64
        
        self.out_conv = nn.Conv2d(base_filters, 1, kernel_size=1)
        self.sigmoid = nn.Sigmoid()
        
    def forward(self, x, time_emb):
        # Encoder
        c1, p1 = self.enc1(x, time_emb)   # c1: (B, 64, 247,247); p1: (B, 64, 123,123)
        c2, p2 = self.enc2(p1, time_emb)    # c2: (B, 128, 123,123); p2: (B, 128, 61,61)
        c3, p3 = self.enc3(p2, time_emb)    # c3: (B, 256, 61,61);  p3: (B, 256, 30,30)
        c4, p4 = self.enc4(p3, time_emb)    # c4: (B, 512, 30,30);  p4: (B, 512, 15,15)
        
        bn = self.bottleneck(p4, time_emb)  # bn: (B, 1024, 15,15)
        
        # Decoder
        d1 = self.dec1(bn, c4, time_emb)    # d1: (B, 512, 30,30)
        d2 = self.dec2(d1, c3, time_emb)     # d2: (B, 256, 60,60)
        d3 = self.dec3(d2, c2, time_emb)     # d3: (B, 128, 120,120)
        d4 = self.dec4(d3, c1, time_emb)     # d4: (B, 64, 240,240)
        
        out = self.out_conv(d4)             # (B, 1, 240,240)
        out = self.sigmoid(out)
        # Resize the output to match the original input size (e.g., 247x247)
        out = F.interpolate(out, size=x.shape[-2:], mode='bilinear', align_corners=False)
        return out

def train_model(model, train_loader, val_loader, criterion, optimizer, num_epochs=10, device='cuda', model_save_path="best_model.pth"):
    model.to(device)
    best_val_loss = float('inf')
    start_time = time.time()
    
    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        for main_x, time_enc, y in train_loader:
            main_x, time_enc, y = main_x.to(device), time_enc.to(device), y.to(device)
            optimizer.zero_grad()
            outputs = model(main_x, time_enc)
            loss = criterion(outputs, y)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * main_x.size(0)
        epoch_loss = running_loss / len(train_loader.dataset)
        
        # Validation
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for main_x, time_enc, y in val_loader:
                main_x, time_enc, y = main_x.to(device), time_enc.to(device), y.to(device)
                outputs = model(main_x, time_enc)
                loss = criterion(outputs, y)
                val_loss += loss.item() * main_x.size(0)
        val_loss /= len(val_loader.dataset)
        
        print(f"Epoch {epoch+1}/{num_epochs}: Train Loss: {epoch_loss:.4f}, Val Loss: {val_loss:.4f}")
        
        # Save the best model
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), model_save_path)
    
    total_time = time.time() - start_time
    print(f"Total training time: {total_time:.2f} seconds")
    return model

def compute_ssim_scores(model, dataloader, device='cuda'):
    model.eval()
    ssim_scores = []
    with torch.no_grad():
        for main_x, time_enc, y in dataloader:
            main_x, time_enc, y = main_x.to(device), time_enc.to(device), y.to(device)
            preds = model(main_x, time_enc)
            # Process each sample in the batch individually.
            for i in range(preds.shape[0]):
                pred_img = preds[i, 0, :, :].cpu().numpy()
                gt_img = y[i, 0, :, :].cpu().numpy()
                score = ssim(gt_img, pred_img, data_range=1.0)
                ssim_scores.append(score)
    return ssim_scores

def average_across_models(ssim_dict):
    # Transpose the list of lists: shape [num_models][num_samples] -> [num_samples][num_models]
    return list(np.mean(list(zip(*ssim_dict.values())), axis=1))

def std_across_models(ssim_dict):
    # Transpose the list of lists: shape [num_models][num_samples] -> [num_samples][num_models]
    return list(np.std(list(zip(*ssim_dict.values())), axis=1))

In [ ]:


EPS = 1e-8

# ----------------- small utils -----------------
def _ensure_dir(p):
    os.makedirs(p, exist_ok=True); return p

def _ensure_3ch(x):
    return x if x.size(1) == 3 else x.repeat(1, 3, 1, 1)

def _per_image_func(func, x, y, **kwargs):
    """Try reduction='none'; else per-image loop; return list[float] len=B."""
    try:
        out = func(x, y, reduction='none', **kwargs)
        return out.detach().cpu().view(-1).tolist()
    except TypeError:
        scores = []
        for b in range(x.size(0)):
            v = func(x[b:b+1], y[b:b+1], **kwargs)
            v = v.detach().cpu().item() if torch.is_tensor(v) else float(v)
            scores.append(float(v))
        return scores

def _nan_list(b):  # helper for missing metrics
    return [float('nan')] * b

def slugify(s: str) -> str:
    return "".join(c if c.isalnum() else "_" for c in s)

# ---------- Compatibility creators for PIQ ----------
def make_lpips_compute(device):
    # 1) Module with net_type
    try:
        metric = piq.LPIPS(net_type='vgg', reduction='none').to(device).eval()
        return lambda a,b: metric(a,b), "LPIPS module (net_type='vgg')"
    except TypeError:
        pass
    except Exception:
        pass
    # 2) Module default net
    try:
        metric = piq.LPIPS(reduction='none').to(device).eval()
        return lambda a,b: metric(a,b), "LPIPS module (default net)"
    except Exception:
        pass
    # 3) Functional
    if hasattr(piq, "lpips"):
        def lpips_fn(a,b):
            return piq.lpips(a, b, net_type='vgg', reduction='none').to(a.device)
        return lpips_fn, "LPIPS functional (net_type='vgg')"
    # Fallback
    def lpips_nan(a,b):
        return torch.full((a.size(0),), float('nan'), device=a.device)
    return lpips_nan, "LPIPS unavailable → NaN"

def make_dists_compute(device):
    try:
        metric = piq.DISTS(reduction='none').to(device).eval()
        return lambda a,b: metric(a,b), "DISTS module (reduction='none')"
    except TypeError:
        pass
    except Exception:
        pass
    try:
        metric = piq.DISTS().to(device).eval()
        def dists_loop(a,b):
            vals = []
            for i in range(a.size(0)):
                v = metric(a[i:i+1], b[i:i+1])
                vals.append(v.detach().to(a.device).view(1))
            return torch.cat(vals, dim=0)
        return dists_loop, "DISTS module (per-image loop)"
    except Exception:
        pass
    def dists_nan(a,b):
        return torch.full((a.size(0),), float('nan'), device=a.device)
    return dists_nan, "DISTS unavailable → NaN"

def make_msssim_compute(device, data_range):
    # Prefer functional (batch)
    if hasattr(piq, "ms_ssim") and callable(getattr(piq, "ms_ssim")):
        def fn(a,b):
            try:
                return piq.ms_ssim(a, b, data_range=data_range, reduction='none')
            except TypeError:
                vals = []
                for i in range(a.size(0)):
                    v = piq.ms_ssim(a[i:i+1], b[i:i+1], data_range=data_range)
                    if torch.is_tensor(v): v = v.detach().to(a.device).view(1)
                    else: v = torch.tensor(float(v), device=a.device).view(1)
                    vals.append(v)
                return torch.cat(vals, dim=0)
        return fn, "MS-SSIM functional"
    # Try loss classes → score = 1 - loss
    for cls_name in ["MultiScaleSSIMLoss", "MS_SSIMLoss"]:
        if hasattr(piq, cls_name):
            Cls = getattr(piq, cls_name)
            try:
                metric = Cls(data_range=data_range, reduction='none').to(device).eval()
            except TypeError:
                metric = Cls(data_range=data_range).to(device).eval()
            def fn(a,b,metric=metric):
                out = metric(a,b)
                if out.dim() == 0 or out.numel() == 1:
                    vals = []
                    for i in range(a.size(0)):
                        v = metric(a[i:i+1], b[i:i+1])
                        vals.append(v.detach().to(a.device).view(1))
                    out = torch.cat(vals, dim=0)
                return (1.0 - out).view(-1)
            return fn, f"{cls_name} (1 - loss)"
    # Torchmetrics fallback
    try:
        from torchmetrics.image.ssim import MultiScaleStructuralSimilarityIndexMeasure as TM_MSSSIM
        metric = TM_MSSSIM(data_range=data_range).to(device).eval()
        def fn(a,b,metric=metric):
            vals = []
            for i in range(a.size(0)):
                v = metric(a[i:i+1], b[i:i+1])
                vals.append(v.detach().to(a.device).view(1))
            return torch.cat(vals, dim=0).view(-1)
        return fn, "torchmetrics.MS-SSIM"
    except Exception:
        pass
    def nan_fn(a,b):
        return torch.full((a.size(0),), float('nan'), device=a.device)
    return nan_fn, "MS-SSIM unavailable → NaN"

def make_ssim_compute(device, data_range):
    if hasattr(piq, "ssim") and callable(getattr(piq, "ssim")):
        def fn(a,b):
            try:
                return piq.ssim(a, b, data_range=data_range, reduction='none')
            except TypeError:
                vals = []
                for i in range(a.size(0)):
                    v = piq.ssim(a[i:i+1], b[i:i+1], data_range=data_range)
                    if torch.is_tensor(v): v = v.detach().to(a.device).view(1)
                    else: v = torch.tensor(float(v), device=a.device).view(1)
                    vals.append(v)
                return torch.cat(vals, dim=0)
        return fn, "SSIM functional"
    for cls_name in ["SSIMLoss", "SSIM"]:
        if hasattr(piq, cls_name):
            Cls = getattr(piq, cls_name)
            try:
                metric = Cls(data_range=data_range, reduction='none').to(device).eval()
            except TypeError:
                metric = Cls(data_range=data_range).to(device).eval()
            def fn(a,b,metric=metric):
                out = metric(a,b)
                if out.dim() == 0 or out.numel() == 1:
                    vals = []
                    for i in range(a.size(0)):
                        v = metric(a[i:i+1], b[i:i+1])
                        vals.append(v.detach().to(a.device).view(1))
                    out = torch.cat(vals, dim=0)
                return (1.0 - out).view(-1) if "Loss" in metric.__class__.__name__ else out.view(-1)
            return fn, f"{cls_name} ({'1 - loss' if 'Loss' in cls_name else 'score'})"
    try:
        from torchmetrics.image.ssim import StructuralSimilarityIndexMeasure as TM_SSIM
        metric = TM_SSIM(data_range=data_range).to(device).eval()
        def fn(a,b,metric=metric):
            vals = []
            for i in range(a.size(0)):
                v = metric(a[i:i+1], b[i:i+1])
                vals.append(v.detach().to(a.device).view(1))
            return torch.cat(vals, dim=0).view(-1)
        return fn, "torchmetrics.SSIM"
    except Exception:
        pass
    def nan_fn(a,b):
        return torch.full((a.size(0),), float('nan'), device=a.device)
    return nan_fn, "SSIM unavailable → NaN"

# ------------- Core evaluator (saves CSV/JSON) -------------
def compute_all_metrics_and_save(
    model, dataloader, device='cuda', data_range=1.0, compute_ssim=True,
    save_dir='.', save_prefix='metrics'
):
    _ensure_dir(save_dir)
    model.eval()

    results = {
        "mae": [], "r2": [], "psnr": [], "soft_dice": [],
        "ms_ssim": [], "fsim": [], "gmsd": [], "vif": [],
        "lpips": [], "dists": [],
    }
    if compute_ssim:
        results["ssim"] = []

    # Robust backends
    lpips_compute, lpips_backend = make_lpips_compute(device)
    dists_compute, dists_backend = make_dists_compute(device)
    msssim_compute, msssim_backend = make_msssim_compute(device, data_range)
    if compute_ssim:
        ssim_compute, ssim_backend = make_ssim_compute(device, data_range)
    else:
        ssim_compute, ssim_backend = (None, "disabled")

    with torch.no_grad():
        for main_x, time_enc, y in dataloader:
            main_x, time_enc, y = main_x.to(device), time_enc.to(device), y.to(device)
            preds = model(main_x, time_enc).clamp(0.0, 1.0)  # [B,1,H,W]

            # MAE
            mae_b = (preds - y).abs().flatten(1).mean(dim=1)
            results["mae"] += mae_b.detach().cpu().tolist()

            # R^2
            p, t = preds.flatten(1), y.flatten(1)
            t_mean = t.mean(dim=1, keepdim=True)
            sse = ((t - p) ** 2).sum(dim=1)
            sst = ((t - t_mean) ** 2).sum(dim=1)
            r2_b = 1.0 - sse / (sst + EPS)
            r2_b = torch.where(sst < 1e-12, (sse < 1e-12).float(), r2_b)
            results["r2"] += r2_b.detach().cpu().tolist()

            # PSNR
            mse_b = F.mse_loss(preds, y, reduction='none').flatten(1).mean(dim=1)
            psnr_b = 10.0 * torch.log10((data_range ** 2) / (mse_b + EPS))
            results["psnr"] += psnr_b.detach().cpu().tolist()

            # Soft Dice
            dims = (1,2,3)
            inter = (preds * y).sum(dim=dims)
            denom = preds.pow(2).sum(dim=dims) + y.pow(2).sum(dim=dims)
            dice_b = (2.0 * inter + 1e-6) / (denom + 1e-6)
            results["soft_dice"] += dice_b.detach().cpu().tolist()

            # MS-SSIM (robust)
            msssim_b = msssim_compute(preds, y).detach().cpu().view(-1).tolist()
            results["ms_ssim"] += msssim_b

            # FSIM (chromatic=False if grayscale)
            chromatic = preds.size(1) == 3
            fsim_fn = getattr(piq, 'fsim', None)
            if fsim_fn is None:
                results["fsim"] += _nan_list(preds.size(0))
            else:
                results["fsim"] += _per_image_func(fsim_fn, preds, y, data_range=data_range, chromatic=chromatic)

            # GMSD
            gmsd_fn = getattr(piq, 'gmsd', None)
            if gmsd_fn is None:
                results["gmsd"] += _nan_list(preds.size(0))
            else:
                results["gmsd"] += _per_image_func(gmsd_fn, preds, y, data_range=data_range)

            # VIF / VIF-P
            vif_fn = getattr(piq, 'vif_p', None) or getattr(piq, 'vif', None)
            if vif_fn is None:
                results["vif"] += _nan_list(preds.size(0))
            else:
                results["vif"] += _per_image_func(vif_fn, preds, y, data_range=data_range)

            # LPIPS (3ch)
            preds_rgb, y_rgb = _ensure_3ch(preds), _ensure_3ch(y)
            lpips_b = lpips_compute(preds_rgb, y_rgb)
            results["lpips"] += lpips_b.view(-1).detach().cpu().tolist()

            # DISTS (3ch)
            dists_b = dists_compute(preds_rgb, y_rgb)
            results["dists"] += dists_b.view(-1).detach().cpu().tolist()

            # Optional SSIM
            if compute_ssim:
                ssim_b = ssim_compute(preds, y).detach().cpu().view(-1).tolist()
                results["ssim"] += ssim_b

    # Summaries
    summary = {}
    for k, v in results.items():
        arr = np.asarray(v, dtype=np.float64)
        summary[k] = {"mean": float(np.nanmean(arr)), "std": float(np.nanstd(arr))}

    # Save
    per_image_csv = os.path.join(save_dir, f"{save_prefix}_per_image.csv")
    summary_csv   = os.path.join(save_dir, f"{save_prefix}_summary.csv")
    summary_json  = os.path.join(save_dir, f"{save_prefix}_summary.json")

    keys = list(results.keys())
    N = len(next(iter(results.values()))) if keys else 0
    with open(per_image_csv, "w", newline="") as f:
        w = csv.writer(f); w.writerow(["index"] + keys)
        for idx in range(N):
            w.writerow([idx] + [results[k][idx] for k in keys])

    with open(summary_csv, "w", newline="") as f:
        w = csv.writer(f); w.writerow(["metric", "mean", "std"])
        for k, stats in summary.items():
            w.writerow([k, stats["mean"], stats["std"]])

    with open(summary_json, "w") as f:
        json.dump(summary, f, indent=2)

    return results, summary, {
        "per_image_csv": per_image_csv,
        "summary_csv": summary_csv,
        "summary_json": summary_json,
        "lpips_backend": lpips_compute.__name__ if hasattr(lpips_compute, "__name__") else "LPIPS backend",
        "dists_backend": dists_compute.__name__ if hasattr(dists_compute, "__name__") else "DISTS backend",
        "msssim_backend": "auto",
        "ssim_backend": ssim_backend,
    }

# ------------- Aggregation helpers -------------
def average_across_models(model_to_scores: dict):
    if not model_to_scores: return []
    arrs = [np.asarray(v, dtype=np.float64) for v in model_to_scores.values()]
    Ls = [a.size for a in arrs]
    assert len(set(Ls)) == 1, f"Score length mismatch across models: {Ls}"
    return np.vstack(arrs).mean(axis=0).tolist()

def std_across_models(model_to_scores: dict):
    if not model_to_scores: return []
    arrs = [np.asarray(v, dtype=np.float64) for v in model_to_scores.values()]
    Ls = [a.size for a in arrs]
    assert len(set(Ls)) == 1, f"Score length mismatch across models: {Ls}"
    return np.vstack(arrs).std(axis=0).tolist()

# ------------- Runner: prints which best_model is used -------------
def run_all(i=1, SEED=123, batch_size=8, data_range=1.0, compute_ssim=True):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    torch.manual_seed(SEED); random.seed(SEED); np.random.seed(SEED)

    # Load held-out datasets
    train_dataset = torch.load(f"training_data/train_dataset_{i}.pt", weights_only=False)
    val_dataset   = torch.load(f"training_data/val_dataset_{i}.pt",  weights_only=False)
    test_dataset  = torch.load(f"training_data/test_dataset_{i}.pt",  weights_only=False)
    full_dataset  = ConcatDataset([train_dataset, val_dataset, test_dataset])
    full_loader   = DataLoader(full_dataset, batch_size=batch_size, shuffle=False, num_workers=0)

    print(f"\n[Held-out core {i}] Dataset sizes: train={len(train_dataset)}, val={len(val_dataset)}, test={len(test_dataset)}, total={len(full_dataset)}")

    per_model_dir      = _ensure_dir("final_result/per_model_metrics")
    aggregate_plot_dir = _ensure_dir("final_result")
    aggregate_csv_dir  = aggregate_plot_dir

    METRICS = ["mae","r2","psnr","soft_dice","ms_ssim","fsim","gmsd","vif","lpips","dists"]
    if compute_ssim: METRICS.append("ssim")
    groups = {f"core{r}": {m: {} for m in METRICS} for r in range(1, 8)}
    len_full = len(full_dataset)

    def eval_and_record(combo, r):
        label = f"Training with core {', '.join(map(str, combo))}"
        model_dir = f"best_model_core{r}"
        model_file = f"best_model_{'_'.join(map(str, combo))}.pth"
        model_path = os.path.join(model_dir, model_file)

        print(f"[Held-out core {i}] Testing model: {model_path} on {len_full} samples (batch_size={batch_size})")
        if not os.path.exists(model_path):
            print("  -> SKIP (file not found)")
            return

        model = ConditionalUNet(input_channels=11, time_channels=1, base_filters=64)
        state = torch.load(model_path, map_location=device)
        model.load_state_dict(state); model.to(device)

        save_prefix = f"core{i}_G{r}_{slugify(label)}"
        results, summary, paths = compute_all_metrics_and_save(
            model=model,
            dataloader=full_loader,
            device=device,
            data_range=data_range,
            compute_ssim=compute_ssim,
            save_dir=per_model_dir,
            save_prefix=save_prefix
        )
        print(f"  -> Saved per-image: {paths['per_image_csv']}")
        print(f"  -> Saved summary : {paths['summary_csv']}")
        if 'ms_ssim' in summary and 'psnr' in summary:
            print(f"  -> MS-SSIM mean={summary['ms_ssim']['mean']:.4f}, PSNR mean={summary['psnr']['mean']:.2f} dB")

        for m in METRICS:
            groups[f"core{r}"][m][label] = results[m]

        del model, state
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()

    # Enumerate combos excluding held-out i
    all_cores = list(range(1, 9))
    other_cores = [c for c in all_cores if c != i]
    for r in range(1, 8):
        for combo in combinations(other_cores, r):
            eval_and_record(combo, r)

    # Aggregate per group & plot
    labels_by_group = ['1 core','2 cores','3 cores','4 cores','5 cores','6 cores','7 cores']
    aggregated_summary = []
    for m in METRICS:
        avg_lists, std_lists = [], []
        for r in range(1, 8):
            avg = average_across_models(groups[f"core{r}"][m])
            std = std_across_models(groups[f"core{r}"][m])
            avg_lists.append(avg); std_lists.append(std)

            arr_avg, arr_std = np.asarray(avg), np.asarray(std)
            aggregated_summary.append({
                "metric": m,
                "group": labels_by_group[r-1],
                "mean_avg": float(np.nanmean(arr_avg)) if arr_avg.size else float('nan'),
                "std_avg":  float(np.nanstd(arr_avg))  if arr_avg.size else float('nan'),
                "mean_std_across_models": float(np.nanmean(arr_std)) if arr_std.size else float('nan'),
                "std_of_std_across_models": float(np.nanstd(arr_std)) if arr_std.size else float('nan'),
            })

        if any(len(x) for x in avg_lists):
            plt.figure(figsize=(8, 6))
            plt.boxplot(avg_lists, labels=labels_by_group)
            ylabel = m.upper() if m not in ("ms_ssim","soft_dice") else m.replace("_","-").upper()
            plt.ylabel(ylabel)
            plt.title(f"{m.upper()} Box Plot (Per-image mean across models) for Core {i}")
            plt.tight_layout()
            plt.savefig(os.path.join(aggregate_plot_dir, f"{m.upper()}_Test_Avg_Core{i}.png"))
            plt.close()

        if any(len(x) for x in std_lists):
            plt.figure(figsize=(8, 6))
            plt.boxplot(std_lists, labels=labels_by_group)
            plt.ylabel("STD across models")
            plt.title(f"STD of {m.upper()} (Per-image) for Core {i}")
            plt.tight_layout()
            plt.savefig(os.path.join(aggregate_plot_dir, f"{m.upper()}_Test_Std_Core{i}.png"))
            plt.close()

    # Save aggregated summary
    agg_csv  = os.path.join(aggregate_csv_dir, f"Aggregated_Summary_Core{i}.csv")
    agg_json = os.path.join(aggregate_csv_dir, f"Aggregated_Summary_Core{i}.json")
    with open(agg_csv, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=["metric","group","mean_avg","std_avg","mean_std_across_models","std_of_std_across_models"])
        writer.writeheader(); writer.writerows(aggregated_summary)
    with open(agg_json, "w") as f:
        json.dump(aggregated_summary, f, indent=2)

    print(f"[Held-out core {i}] Aggregated plots saved to: {aggregate_plot_dir}")
    print(f"[Held-out core {i}] Aggregated summary saved to: {agg_csv} and {agg_json}")



In [ ]:

for i in range(1, 9):   # iterate held-out core i = 1..8
    print(f"\n==== Running evaluation for held-out core {i} ====")
    run_all(
        i=i,
        SEED=SEED,
        batch_size=8,
        data_range=1.0,
        compute_ssim=True,   # set False if you don’t want SSIM
    )